In [1]:
import glob
import math
import os
import re

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity, rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tqdm import tqdm

# Gene Exp

In [2]:
gene_exp = pd.read_csv("../nci_data/nci60_gene_exp.csv", index_col=0).T
gene_exp.head()

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
MCF7,1.92,2.89,0.03,0.03,0.21,0.00,0.0,0.00,2.46,0.0,...,8.52,22.48,0.44,1.51,5.59,0.97,0.94,6.99,2.52,3.05
MDA_MB_231,0.49,0.12,0.02,0.02,0.00,0.01,0.0,0.07,0.22,0.0,...,16.76,27.56,0.32,3.43,1.44,0.55,1.58,8.15,0.45,18.57
HS578T,3.37,1.17,0.04,0.04,0.00,0.09,0.0,0.00,0.35,0.3,...,1.97,3.22,0.16,0.34,2.11,0.14,1.42,103.53,1.27,1.65
BT_549,6.00,1.92,0.00,0.00,0.00,0.00,0.0,0.00,5.52,0.0,...,1.93,3.75,0.45,0.94,2.79,0.49,1.22,36.05,2.50,2.31
T47D,3.73,1.65,0.01,0.01,0.06,0.00,0.0,0.15,0.71,0.0,...,4.02,25.63,0.55,1.85,5.18,0.80,1.51,13.00,1.64,1.28


In [3]:
# nsc_class = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
#     ["NSC", "MECHANISM"]
# ]
# nsc_class = nsc_class[nsc_class.MECHANISM == "DNA"]
# nsc_class.shape

# Drug Response

In [4]:
drug_response = pd.read_csv("../nci_data/nci60Act.csv", index_col=0)
drug_response.columns = gene_exp.index
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,-0.271314,-0.303539,-0.815183,-0.231499,1.934731,-0.357577,-0.719253,-0.380720,-1.281589,-0.175915,...,-0.403983,-0.434885,-0.519867,-1.648059,1.657273,-0.269717,0.002290,-0.390689,-0.379420,1.059515
17,-0.354110,-0.304675,-0.222024,1.483613,1.509397,0.335572,0.424922,1.140166,-0.941890,0.330808,...,1.302516,-0.941890,0.521200,-0.329639,-0.941890,0.717851,-0.239018,-0.315533,-0.761559,1.120673
89,NaN,NaN,NaN,NaN,NaN,-0.184194,-1.429903,-0.165433,-1.429903,-0.216723,...,NaN,NaN,0.368433,-0.655707,-0.144006,0.743888,-0.435620,-0.184805,-0.101929,-0.118807
185,NaN,NaN,NaN,NaN,NaN,0.539343,NaN,0.230402,-0.765829,-1.125208,...,NaN,NaN,-0.061612,NaN,NaN,1.675353,0.918722,1.391604,-0.904724,0.918522
295,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,...,-0.264586,-0.264586,4.822657,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900911,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,...,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,1.594059,-0.167151,-0.167151,-0.167151
900922,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,...,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786
900964,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,...,-0.158754,NaN,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754
900974,-0.132453,-0.132453,-0.132453,NaN,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,...,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,NaN,-0.132453,-0.132453,-0.132453


In [5]:
l = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[["NSC", "SMILES"]]
l

,NSC,SMILES
0,1,CC1=CC(=O)C=CC1=O
1,17,CCCCCCCCCCCCCCCC1=C(C=CC(=C1)O)N
2,89,CN(C)CCC(=O)C1=CC=CC=C1.Cl
3,185,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
4,758187,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
...,...,...
23186,799649,C1CC2=C(C1)SC3=NC=NC(=C23)NC4=C(NN=C4)C(=O)NC5...
23187,803209,CCN1C2C(N(C1=O)CC)N3C(=O)C(=C4C5=CC=CC=C5N(C4=...
23188,803778,C1=CC=C(C=C1)C=NC2=C(C(C3=C(O2)C4=CC=CC=C4C=C3...
23189,807047,CC1=C(C=C(C=C1)NC(=O)C2=CC(=NC=C2)C(F)(F)F)C3=...


In [6]:
drug_response = drug_response[drug_response.index.isin(l["NSC"])]
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,-0.271314,-0.303539,-0.815183,-0.231499,1.934731,-0.357577,-0.719253,-0.380720,-1.281589,-0.175915,...,-0.403983,-0.434885,-0.519867,-1.648059,1.657273,-0.269717,0.002290,-0.390689,-0.379420,1.059515
17,-0.354110,-0.304675,-0.222024,1.483613,1.509397,0.335572,0.424922,1.140166,-0.941890,0.330808,...,1.302516,-0.941890,0.521200,-0.329639,-0.941890,0.717851,-0.239018,-0.315533,-0.761559,1.120673
89,NaN,NaN,NaN,NaN,NaN,-0.184194,-1.429903,-0.165433,-1.429903,-0.216723,...,NaN,NaN,0.368433,-0.655707,-0.144006,0.743888,-0.435620,-0.184805,-0.101929,-0.118807
185,NaN,NaN,NaN,NaN,NaN,0.539343,NaN,0.230402,-0.765829,-1.125208,...,NaN,NaN,-0.061612,NaN,NaN,1.675353,0.918722,1.391604,-0.904724,0.918522
295,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,...,-0.264586,-0.264586,4.822657,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,-0.372382,-0.372382,1.025754,-0.372382,-0.372382,-0.372382,-0.372382,-0.372382,-0.372382,-0.372382,...,-0.372382,-0.372382,-0.372382,3.560518,-0.372382,NaN,0.207451,0.453329,-0.372382,-0.372382
831147,1.150515,-1.003232,0.363190,-0.003700,-0.279367,-0.769224,1.026660,0.282876,-0.012740,0.398150,...,-0.291960,-0.413240,-0.403203,0.490038,-0.553374,0.860195,0.410215,-0.460626,-3.036585,-0.235431
831270,1.891309,-0.745408,-0.226427,-0.432638,-0.475326,-0.660433,-0.351529,-0.494257,-0.666792,-0.464728,...,-0.339531,-0.920607,-0.571579,1.118293,1.525184,2.375363,-0.373980,-0.686271,-0.539730,1.604489
831271,0.446573,-0.176892,-0.118513,0.244076,0.345654,-0.395343,-0.140526,-0.067327,-1.587690,-0.064476,...,0.112988,-0.416629,0.156786,0.347218,-0.310631,0.097501,0.355559,-0.106126,-0.326793,0.530565


In [7]:
df = pd.DataFrame()
for i in drug_response.columns:
    tmp = pd.DataFrame(drug_response[i]).reset_index().dropna()
    tmp["cell"] = [i] * len(tmp)
    tmp.columns = ["Drug", "Value", "Cell"]
    tmp = tmp[["Drug", "Cell", "Value"]]
    df = pd.concat([df, tmp])
df

,Drug,Cell,Value
0,1,MCF7,-0.271314
1,17,MCF7,-0.354110
4,295,MCF7,-0.264586
5,353,MCF7,0.744838
6,384,MCF7,-0.041323
...,...,...,...
23186,830912,UO_31,-0.372382
23187,831147,UO_31,-0.235431
23188,831270,UO_31,1.604489
23189,831271,UO_31,0.530565


# Zero shot prediction

In [8]:
unique_drugs = df["Drug"].unique()
unique_cells = df["Cell"].unique()

# Split drugs and cell lines into training, validation, and test sets
train_drugs, test_val_drugs = train_test_split(
    unique_drugs, test_size=0.5, random_state=42
)
val_drugs, test_drugs = train_test_split(test_val_drugs, test_size=0.5, random_state=42)

train_cells, test_val_cells = train_test_split(
    unique_cells, test_size=0.55, random_state=42
)
val_cells, test_cells = train_test_split(test_val_cells, test_size=0.5, random_state=42)

# Split the dataset
train_df = df[(df["Drug"].isin(train_drugs)) & (df["Cell"].isin(train_cells))]
val_df = df[(df["Drug"].isin(val_drugs)) & (df["Cell"].isin(val_cells))]
test_df = df[(df["Drug"].isin(test_drugs)) & (df["Cell"].isin(test_cells))]


# Function to balance label distribution
def balance_labels(df, threshold=0.5):
    positive = df[df["Value"] > 0]
    negative = df[df["Value"] <= 0]
    min_count = min(len(positive), len(negative))
    balanced_positive = positive.sample(min_count, random_state=42)
    balanced_negative = negative.sample(min_count, random_state=42)
    return pd.concat([balanced_positive, balanced_negative])


# Balance label distribution across all sets
train_df = balance_labels(train_df)
val_df = balance_labels(val_df)
test_df = balance_labels(test_df)

# Separate features and labels
X_train = train_df[["Drug", "Cell"]]
y_train = np.array(train_df["Value"] > 0, dtype=float)

X_val = val_df[["Drug", "Cell"]]
y_val = np.array(val_df["Value"] > 0, dtype=float)

X_test = test_df[["Drug", "Cell"]]
y_test = np.array(test_df["Value"] > 0, dtype=float)

# Calculate total samples
total_samples = len(X_train) + len(X_val) + len(X_test)


# Function to calculate and format label ratios
def get_label_ratio(y):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    ratio_str = ", ".join(
        [f"Label {label}: {count/total:.2%}" for label, count in zip(unique, counts)]
    )
    return ratio_str


# Display results with percentages and label ratios
print("Train:")
print(X_train.shape, y_train.shape)
print(f"Percentage: {len(X_train)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_train)}")

print("\nValidation:")
print(X_val.shape, y_val.shape)
print(f"Percentage: {len(X_val)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_val)}")

print("\nTest:")
print(X_test.shape, y_test.shape)
print(f"Percentage: {len(X_test)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_test)}")

print(f"\nTotal samples: {total_samples}")
print(
    f"Ratio (Train:Validation:Test): {len(X_train):.0f}:{len(X_val):.0f}:{len(X_test):.0f}"
)
print(
    f"Overall Label Ratio: {get_label_ratio(np.concatenate([y_train, y_val, y_test]))}"
)

# X_train.to_csv("../nci_data/train.csv", index=False)
# X_test.to_csv("../nci_data/test.csv", index=False)
# X_val.to_csv("../nci_data/val.csv", index=False)

# np.save("../nci_data/train_labels.npy", y_train)
# np.save("../nci_data/test_labels.npy", y_test)
# np.save("../nci_data/val_labels.npy", y_val)

Train:
(268192, 2) (268192,)
Percentage: 65.78%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Validation:
(59740, 2) (59740,)
Percentage: 14.65%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Test:
(79804, 2) (79804,)
Percentage: 19.57%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Total samples: 407736
Ratio (Train:Validation:Test): 268192:59740:79804
Overall Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%


In [9]:
drug_response = drug_response.fillna(0)
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,-0.271314,-0.303539,-0.815183,-0.231499,1.934731,-0.357577,-0.719253,-0.380720,-1.281589,-0.175915,...,-0.403983,-0.434885,-0.519867,-1.648059,1.657273,-0.269717,0.002290,-0.390689,-0.379420,1.059515
17,-0.354110,-0.304675,-0.222024,1.483613,1.509397,0.335572,0.424922,1.140166,-0.941890,0.330808,...,1.302516,-0.941890,0.521200,-0.329639,-0.941890,0.717851,-0.239018,-0.315533,-0.761559,1.120673
89,0.000000,0.000000,0.000000,0.000000,0.000000,-0.184194,-1.429903,-0.165433,-1.429903,-0.216723,...,0.000000,0.000000,0.368433,-0.655707,-0.144006,0.743888,-0.435620,-0.184805,-0.101929,-0.118807
185,0.000000,0.000000,0.000000,0.000000,0.000000,0.539343,0.000000,0.230402,-0.765829,-1.125208,...,0.000000,0.000000,-0.061612,0.000000,0.000000,1.675353,0.918722,1.391604,-0.904724,0.918522
295,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,...,-0.264586,-0.264586,4.822657,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,-0.372382,-0.372382,1.025754,-0.372382,-0.372382,-0.372382,-0.372382,-0.372382,-0.372382,-0.372382,...,-0.372382,-0.372382,-0.372382,3.560518,-0.372382,0.000000,0.207451,0.453329,-0.372382,-0.372382
831147,1.150515,-1.003232,0.363190,-0.003700,-0.279367,-0.769224,1.026660,0.282876,-0.012740,0.398150,...,-0.291960,-0.413240,-0.403203,0.490038,-0.553374,0.860195,0.410215,-0.460626,-3.036585,-0.235431
831270,1.891309,-0.745408,-0.226427,-0.432638,-0.475326,-0.660433,-0.351529,-0.494257,-0.666792,-0.464728,...,-0.339531,-0.920607,-0.571579,1.118293,1.525184,2.375363,-0.373980,-0.686271,-0.539730,1.604489
831271,0.446573,-0.176892,-0.118513,0.244076,0.345654,-0.395343,-0.140526,-0.067327,-1.587690,-0.064476,...,0.112988,-0.416629,0.156786,0.347218,-0.310631,0.097501,0.355559,-0.106126,-0.326793,0.530565


# Masking

In [10]:
# Validation Mask
for _, row in X_val.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# Test Mask
for _, row in X_test.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# DTI

In [11]:
dti = pd.read_csv("../data/full_table.csv")
dti = dti.dropna(subset="NSC").reset_index(drop=True)
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,APOD,122759.0
1,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,PTGDS,122759.0
2,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH12,122759.0
3,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH5,122759.0
4,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH13,122759.0


In [12]:
print("unique drugs:", len(set(dti.NSC)))
print("unique genes:", len(set(dti.Gene)))

unique drugs: 392
unique genes: 689


In [13]:
len(set(drug_response.index) & set(int(i) for i in set(dti.NSC)))

392

In [14]:
len(set(gene_exp.columns) & set(dti.Gene))

633

## All drugs are in drug response.

In [15]:
dti = dti[dti.Gene.isin(set(gene_exp.columns) & set(dti.Gene))]
dti

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,APOD,122759.0
1,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,PTGDS,122759.0
2,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH12,122759.0
3,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH5,122759.0
4,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH13,122759.0
...,...,...,...,...,...,...,...
1509,Trilaciclib,DB15442,NaN,NaN,CN1CCN(CC1)C1=CN=C(NC2=NC3=C(C=C4N3C3(CCCCC3)C...,CDK7,816987.0
1511,Rocaglamide,DB15495,NaN,NaN,COC1=CC=C(C=C1)[C@@]12OC3=C(C(OC)=CC(OC)=C3)[C...,PHB2,326408.0
1512,BOS172722,DB15498,NaN,NaN,CCOC1=CC(=CC=C1NC1=NC=C2C=C(C)N=C(NCC(C)(C)C)C...,TTK,809694.0
1513,Sotorasib,DB15569,NaN,NaN,CC(C)C1=NC=CC(C)=C1N1C(=O)N=C(N2CCN(C[C@@H]2C)...,KRAS,818433.0


# Selected highly variable genes

In [16]:
print(
    "Density: ",
    round((len(gene_exp.values.nonzero()[0]) / gene_exp.size) * 100, 2),
    "%",
)

Density:  75.09 %


In [17]:
variance = gene_exp.std()
variance = variance.sort_values(ascending=False)
variance = pd.DataFrame(variance > np.percentile(variance, 90))
variance = list(variance[variance[0] == True].index)
len(variance)

2383

In [18]:
print("DTI unique genes: ", len(set(dti["Gene"])))
print("Top 90% variable genes: ", len(variance))
print("Total: ", len(set(variance) | (set(dti["Gene"]))))

DTI unique genes:  633
Top 90% variable genes:  2383
Total:  2881


# Preprocessed data dims

In [19]:
genes = sorted(list(set(variance) | (set(dti["Gene"]))))
gene_exp = gene_exp[genes]
gene_exp.shape

(60, 2881)

In [20]:
drug_response.shape

(23191, 60)

# Normalize

In [21]:
gene_norm_cell = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp),
    index=gene_exp.index,
    columns=gene_exp.columns,
)
gene_norm_cell

,A2M,AAK1,ABCA1,ABCB1,ABCC1,ABCC5,ABL1,ABL2,ABRACL,ACACB,...,ZNF207,ZNF22,ZNF580,ZNF593,ZNF706,ZNHIT1,ZNHIT3,ZNRD1,ZWINT,ZYX
MCF7,-0.202516,-1.154608,-0.751759,-0.165123,-0.692354,0.854276,-0.271267,-0.770514,-0.387611,2.003560,...,-0.217212,-0.199171,-0.587631,-0.425805,-0.053803,1.583642,0.133603,0.091624,0.544875,-1.000041
MDA_MB_231,-0.203121,0.473487,-0.328914,-0.168799,-0.933710,0.372579,-1.245398,0.740272,1.908655,-0.929580,...,-0.758387,2.746291,-0.997701,-1.261613,3.785998,-0.289811,1.007383,0.971839,0.992416,-0.982226
HS578T,-0.201911,-0.289038,-0.530870,-0.162917,-1.147082,-0.454334,-0.126488,-0.585990,-0.743055,-0.907017,...,-1.285914,-0.223103,-0.134860,-0.544310,-0.551808,-0.490514,-0.908373,-0.549500,-1.151904,0.482635
BT_549,-0.204330,-0.927911,-0.726515,-0.168799,-0.986178,-0.799550,3.612744,2.450958,-0.073691,-0.974705,...,-1.247937,-0.432291,2.169666,1.576875,-0.256067,0.015607,-0.815552,-0.596172,-1.105212,-0.553733
T47D,-0.203725,-0.927911,-0.694959,-0.168064,-1.122597,2.303381,-0.475030,-0.685940,-0.377645,-0.613703,...,0.681187,-0.409245,-0.272454,-0.018873,-0.211746,-0.428760,1.302598,0.501845,0.822386,-0.907739
SF_268,-0.201306,-0.412691,-0.701270,-0.168799,-1.349961,-1.120681,-0.757439,-0.355336,-0.380137,-0.771641,...,-1.039063,-0.270968,-0.051490,0.189922,-0.671878,-0.565694,-0.817645,-0.507741,-0.562524,-0.463888
SF_295,-0.203121,3.111413,0.346375,-0.074688,1.311951,-0.586801,0.143409,-0.043952,-0.189958,-0.365514,...,3.527697,-0.156624,1.714861,2.626495,0.276589,4.171303,2.326429,0.197250,0.646189,2.257423
SF_539,-0.190421,1.318448,-0.625537,-0.168799,0.923682,-0.490461,3.693177,1.489899,-0.282140,-0.252701,...,0.106188,-0.286923,0.502951,-0.247106,-0.430128,-0.528776,-0.825322,-0.080325,-0.539619,0.996058
SNB_19,-0.180140,-0.577562,0.762909,-0.165123,-0.961693,-0.542645,0.161283,-0.347647,-0.568655,-0.546015,...,-0.858671,0.007357,0.376203,-0.550580,-0.646897,0.617716,-0.403088,-0.617460,-1.280528,-0.639739
SNB_75,7.334074,0.968098,0.838642,-0.168064,0.626360,0.023349,2.238236,0.317406,-0.394255,-0.049638,...,-0.081918,-0.308197,0.101015,-0.648394,-0.382583,0.345189,-0.539878,-0.451243,-0.050671,0.261171


In [22]:
gene_norm_gene = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp.T),
    index=gene_exp.columns,
    columns=gene_exp.index,
).T
gene_norm_gene

,A2M,AAK1,ABCA1,ABCB1,ABCC1,ABCC5,ABL1,ABL2,ABRACL,ACACB,...,ZNF207,ZNF22,ZNF580,ZNF593,ZNF706,ZNHIT1,ZNHIT3,ZNRD1,ZWINT,ZYX
MCF7,-0.319483,-0.314759,-0.319688,-0.319346,-0.293539,-0.283271,-0.277726,-0.306476,-0.275878,-0.310310,...,-0.117818,-0.293470,-0.243294,-0.195239,-0.247264,0.018883,-0.178058,-0.251850,-0.165804,-0.271839
MDA_MB_231,-0.269319,-0.263002,-0.267518,-0.269375,-0.260702,-0.257959,-0.267491,-0.253137,-0.175024,-0.269181,...,-0.212931,-0.166684,-0.255215,-0.255936,-0.108023,-0.209661,-0.177352,-0.212127,-0.193008,-0.246792
HS578T,-0.286375,-0.271223,-0.283285,-0.285976,-0.261653,-0.266238,-0.217592,-0.262749,-0.265640,-0.285976,...,-0.172334,-0.251285,-0.108934,-0.124385,-0.242912,-0.101756,-0.229354,-0.266039,-0.254675,0.745277
BT_549,-0.340023,-0.331105,-0.339644,-0.340023,-0.311750,-0.328638,-0.075695,-0.242205,-0.243438,-0.339549,...,-0.225032,-0.328638,0.151820,0.135502,-0.263457,-0.092394,-0.272755,-0.325697,-0.304444,0.002009
T47D,-0.032194,-0.031964,-0.032174,-0.032194,-0.031554,-0.029983,-0.030959,-0.031664,-0.030580,-0.032145,...,-0.021132,-0.031835,-0.028277,-0.026080,-0.030060,-0.027367,-0.022915,-0.028498,-0.025842,-0.028974
SF_268,-0.027319,-0.027040,-0.027313,-0.027329,-0.026940,-0.027249,-0.026645,-0.026725,-0.026027,-0.027301,...,-0.024192,-0.026723,-0.023504,-0.021713,-0.026745,-0.023831,-0.025913,-0.026809,-0.025341,-0.018925
SF_295,-0.319743,-0.300397,-0.312868,-0.314707,-0.281651,-0.312908,-0.286048,-0.304554,-0.284729,-0.318544,...,0.050307,-0.302596,-0.139435,-0.052578,-0.261146,0.031960,-0.111535,-0.275056,-0.225372,0.555898
SF_539,-0.266373,-0.248714,-0.266556,-0.267778,-0.216206,-0.255741,-0.094791,-0.220056,-0.220911,-0.265517,...,-0.054279,-0.250425,-0.101268,-0.139275,-0.231665,-0.157851,-0.225311,-0.220056,-0.205635,0.569109
SNB_19,-0.313258,-0.304496,-0.293344,-0.316742,-0.286872,-0.298920,-0.232110,-0.287071,-0.275223,-0.314851,...,-0.131249,-0.255907,-0.064539,-0.156041,-0.285180,0.031941,-0.187803,-0.304794,-0.299716,-0.014059
SNB_75,0.173011,-0.114696,-0.115310,-0.121236,-0.103341,-0.113587,-0.073643,-0.110022,-0.106339,-0.120173,...,-0.046258,-0.115121,-0.070928,-0.086721,-0.105914,-0.048052,-0.095197,-0.113516,-0.084148,0.089109


# Make matrices association matrices by setting 0 threshold and min max scaler.

In [23]:
def min_max_scale(data):
    data = data[data > 0].fillna(0)
    scaler = MinMaxScaler(feature_range=(0, 1))
    return pd.DataFrame(
        scaler.fit_transform(data), index=data.index, columns=data.columns
    )

In [24]:
A_dc = min_max_scale(drug_response)
A_dc

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000303,0.000000,0.0,0.139101
17,0.000000,0.000000,0.000000,0.201118,0.205155,0.046117,0.055794,0.149733,0.0,0.043431,...,0.184168,0.0,0.071499,0.000000,0.0,0.095068,0.000000,0.000000,0.0,0.147130
89,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.050543,0.000000,0.0,0.098516,0.000000,0.000000,0.0,0.000000
185,0.000000,0.000000,0.000000,0.000000,0.000000,0.074121,0.000000,0.030258,0.0,0.000000,...,0.000000,0.0,0.000000,0.000000,0.0,0.221873,0.121670,0.202555,0.0,0.120590
295,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.661584,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,0.000000,0.000000,0.136767,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.000000,0.475721,0.0,0.000000,0.000000,0.065984,0.0,0.000000
831147,0.152764,0.000000,0.048425,0.000000,0.000000,0.000000,0.134805,0.037149,0.0,0.000000,...,0.000000,0.0,0.000000,0.065474,0.0,0.113919,0.000000,0.000000,0.0,0.000000
831270,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.000000,0.149415,0.0,0.314578,0.000000,0.000000,0.0,0.210650
831271,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.021508,0.046392,0.0,0.012912,0.047088,0.000000,0.0,0.069657


In [25]:
A_cg = min_max_scale(gene_norm_gene + gene_norm_cell)
A_cg

,A2M,AAK1,ABCA1,ABCB1,ABCC1,ABCC5,ABL1,ABL2,ABRACL,ACACB,...,ZNF207,ZNF22,ZNF580,ZNF593,ZNF706,ZNHIT1,ZNHIT3,ZNRD1,ZWINT,ZYX
MCF7,0.000000,0.000000,0.000000,0.00000,0.000000,0.125364,0.000000,0.000000,0.000000,0.521956,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.381257,0.000000,0.000000,0.074835,0.000000
MDA_MB_231,0.000000,0.074879,0.000000,0.00000,0.000000,0.025165,0.000000,0.207760,0.325266,0.000000,...,0.000000,0.427873,0.000000,0.000000,0.660697,0.000000,0.259824,0.121694,0.157818,0.000000
HS578T,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.217541
BT_549,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.982954,0.942017,0.000000,0.000000,...,0.000000,0.000000,0.545743,0.433397,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
T47D,0.000000,0.000000,0.000000,0.00000,0.000000,0.499124,0.000000,0.000000,0.000000,0.000000,...,0.160511,0.000000,0.000000,0.000000,0.000000,0.000000,0.400578,0.075823,0.157252,0.000000
SF_268,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.042573,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
SF_295,0.000000,1.000000,0.009987,0.00000,0.489147,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.870091,0.000000,0.370356,0.651450,0.002774,1.000000,0.693326,0.000000,0.083077,0.498418
SF_539,0.000000,0.380551,0.000000,0.00000,0.335883,0.000000,1.000000,0.541579,0.000000,0.000000,...,0.012623,0.000000,0.094429,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.277290
SNB_19,0.000000,0.000000,0.139962,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.073267,0.000000,0.000000,0.154560,0.000000,0.000000,0.000000,0.000000
SNB_75,1.000000,0.303592,0.215601,0.00000,0.248309,0.000000,0.601546,0.088448,0.000000,0.000000,...,0.000000,0.000000,0.007073,0.000000,0.000000,0.070692,0.000000,0.000000,0.000000,0.062057


In [26]:
A_dg = (
    pd.DataFrame(
        np.ones([len(A_dc.index), len(A_cg.columns)]),
        index=A_dc.index,
        columns=A_cg.columns,
    )
    / 2
)
for _, i in dti.iterrows():
    A_dg.loc[int(i["NSC"]), i["Gene"]] = 1
A_dg

,A2M,AAK1,ABCA1,ABCB1,ABCC1,ABCC5,ABL1,ABL2,ABRACL,ACACB,...,ZNF207,ZNF22,ZNF580,ZNF593,ZNF706,ZNHIT1,ZNHIT3,ZNRD1,ZWINT,ZYX
1,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
17,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
89,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
185,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
295,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
831147,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
831270,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
831271,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [27]:
A_dg.sum().sum()

33407359.5

In [28]:
print(
    "Drug Density: ",
    round(len(A_dc.values.nonzero()[0]) / drug_response.size, 4) * 100,
    "%",
)
print("Cell Density: ", round(len(A_cg.values.nonzero()[0]) / A_cg.size, 4) * 100, "%")
print("Gene Density: ", round(len(A_dg.values.nonzero()[0]) / A_dg.size, 5) * 100, "%")

Drug Density:  34.339999999999996 %
Cell Density:  29.7 %
Gene Density:  100.0 %


# Similarity

In [29]:
def normalize_similarity_matrix(df, gamma=None):
    similarity_matrix = rbf_kernel(df.values, gamma=gamma)
    scaler = MinMaxScaler()
    normalized_matrix = scaler.fit_transform(similarity_matrix.reshape(-1, 1))
    normalized_df = pd.DataFrame(
        normalized_matrix.reshape(similarity_matrix.shape),
        index=df.index,
        columns=df.index,
    )

    return normalized_df

In [30]:
cell_sim = normalize_similarity_matrix(drug_response.T)
cell_sim.to_csv("../nci_data/cell_sim.csv")
cell_sim

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
MCF7,1.000000,0.226490,0.144951,0.257350,0.260515,0.289270,0.264194,0.274780,0.187035,0.126872,...,0.350153,0.261205,0.312398,0.101323,0.306294,0.167891,0.230801,0.307151,0.151363,0.136522
MDA_MB_231,0.226490,1.000000,0.259805,0.380267,0.254191,0.396605,0.311961,0.304792,0.322320,0.162854,...,0.439078,0.333595,0.394216,0.141304,0.361337,0.207360,0.275284,0.440484,0.289777,0.172254
HS578T,0.144951,0.259805,1.000000,0.287642,0.160093,0.294730,0.256493,0.233424,0.258563,0.171210,...,0.298768,0.233658,0.248231,0.148433,0.214640,0.127117,0.208163,0.263648,0.184417,0.106271
BT_549,0.257350,0.380267,0.287642,1.000000,0.297898,0.453168,0.349966,0.360449,0.379926,0.184360,...,0.469752,0.404766,0.426219,0.167778,0.387327,0.230319,0.306141,0.457749,0.301419,0.200199
T47D,0.260515,0.254191,0.160093,0.297898,1.000000,0.286762,0.232485,0.212890,0.216570,0.129830,...,0.338996,0.259628,0.283083,0.108402,0.287236,0.167863,0.214698,0.301328,0.214344,0.152674
SF_268,0.289270,0.396605,0.294730,0.453168,0.286762,1.000000,0.406906,0.405725,0.453413,0.203232,...,0.490083,0.476579,0.490760,0.160228,0.462747,0.247171,0.309856,0.537871,0.309229,0.206646
SF_295,0.264194,0.311961,0.256493,0.349966,0.232485,0.406906,1.000000,0.358093,0.388562,0.186934,...,0.465873,0.369241,0.384748,0.188109,0.367736,0.220016,0.263133,0.387361,0.225385,0.170654
SF_539,0.274780,0.304792,0.233424,0.360449,0.212890,0.405725,0.358093,1.000000,0.311755,0.180937,...,0.395713,0.362645,0.433869,0.137974,0.365964,0.201764,0.285767,0.393031,0.202411,0.157511
SNB_19,0.187035,0.322320,0.258563,0.379926,0.216570,0.453413,0.388562,0.311755,1.000000,0.171065,...,0.401784,0.406043,0.370460,0.158497,0.326735,0.173690,0.222024,0.438033,0.269108,0.144802
SNB_75,0.126872,0.162854,0.171210,0.184360,0.129830,0.203232,0.186934,0.180937,0.171065,1.000000,...,0.212008,0.165907,0.189800,0.094427,0.167520,0.097092,0.176557,0.186150,0.109180,0.086736


In [31]:
print("Min:", np.min(cell_sim.values))
print("Max:", np.max(cell_sim.values))
print("Mean:", np.mean(cell_sim.values))
print("Median:", np.median(cell_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.24522297666636753
Median: 0.23436236874623498


In [32]:
gene_sim = normalize_similarity_matrix(gene_norm_cell.T)
gene_sim.to_csv("../nci_data/gene_sim.csv")
gene_sim

,A2M,AAK1,ABCA1,ABCB1,ABCC1,ABCC5,ABL1,ABL2,ABRACL,ACACB,...,ZNF207,ZNF22,ZNF580,ZNF593,ZNF706,ZNHIT1,ZNHIT3,ZNRD1,ZWINT,ZYX
A2M,1.000000,0.154919,0.118251,0.083704,0.107213,0.097999,0.185971,0.137962,0.075351,0.141490,...,0.087766,0.073555,0.094925,0.067556,0.081177,0.107482,0.064276,0.070446,0.079705,0.095941
AAK1,0.154919,1.000000,0.221398,0.075283,0.295840,0.120819,0.172540,0.307975,0.091890,0.176954,...,0.274311,0.102590,0.138336,0.095220,0.085876,0.225797,0.102608,0.096151,0.159984,0.293468
ABCA1,0.118251,0.221398,1.000000,0.073662,0.133881,0.062957,0.179728,0.184854,0.086652,0.086901,...,0.104275,0.058635,0.043566,0.040813,0.063583,0.084409,0.061258,0.046402,0.055517,0.200123
ABCB1,0.083704,0.075283,0.073662,1.000000,0.182178,0.236869,0.143022,0.112365,0.112584,0.101767,...,0.175342,0.187237,0.387115,0.181447,0.155374,0.082825,0.161196,0.231771,0.247777,0.475793
ABCC1,0.107213,0.295840,0.133881,0.182178,1.000000,0.101742,0.191224,0.120523,0.100523,0.109076,...,0.242124,0.101258,0.161771,0.078078,0.053632,0.113069,0.104610,0.115572,0.199311,0.294290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNHIT1,0.107482,0.225797,0.084409,0.082825,0.113069,0.249828,0.082107,0.084987,0.194263,0.126108,...,0.378976,0.209925,0.312362,0.354787,0.102325,1.000000,0.336090,0.242685,0.225571,0.150696
ZNHIT3,0.064276,0.102608,0.061258,0.161196,0.104610,0.406367,0.050881,0.051409,0.490514,0.061603,...,0.502496,0.455127,0.312090,0.270421,0.325852,0.336090,1.000000,0.488875,0.510626,0.091437
ZNRD1,0.070446,0.096151,0.046402,0.231771,0.115572,0.583021,0.116130,0.060280,0.598828,0.067102,...,0.474944,0.827033,0.428108,0.234219,0.265206,0.242685,0.488875,1.000000,0.686624,0.126608
ZWINT,0.079705,0.159984,0.055517,0.247777,0.199311,0.576983,0.102856,0.066065,0.502389,0.123035,...,0.533473,0.580833,0.373112,0.179470,0.282930,0.225571,0.510626,0.686624,1.000000,0.177601


In [33]:
print("Min:", np.min(gene_sim.values))
print("Max:", np.max(gene_sim.values))
print("Mean:", np.mean(gene_sim.values))
print("Median:", np.median(gene_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.16567184552581535
Median: 0.11328310752537764


# NSC to SMILES

In [34]:
convert = dict(
    pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
        ["NSC", "SMILES"]
    ].values
)
SMILES = [convert[i] for i in drug_response.index]
SMILES

['CC1=CC(=O)C=CC1=O',
 'CCCCCCCCCCCCCCCC1=C(C=CC(=C1)O)N',
 'CN(C)CCC(=O)C1=CC=CC=C1.Cl',
 'CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C',
 'C1=CC=C(C=C1)CCCC(=O)O',
 'CCN(CC)CCCNC1=C2C=C(C=CC2=NC3=C1C=CC(=C3)Cl)OC.Cl',
 'CCCCCN(CCCCC)CCCNC1=C2CCCCC2=NC3=C1C=C(C=C3)Cl.OP(=O)(O)O',
 'CC1=C(C(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Br)[N+](=O)[O-]',
 'C1C(O1)C2CO2',
 'C1=CC=C2C(=C1)C(=C(N2)O)N=NC(=S)N',
 'C1=CC(=CC=C1C(=O)NC(CCC(=O)O)C(=O)O)NCC2=CN=C3C(=N2)C(=NC(=N3)N)N',
 'CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C(=O)NC(CCC(=O)O)C(=O)O',
 'C(C(C(=O)O)N)OC(=O)C=[N+]=N',
 'C12=NNN=C1N=C(NC2=O)N',
 'C1=NC2=C(N1)C(=S)N=C(N2)N',
 'C1=NC2=C(N1)C(=S)N=CN2',
 'CC(=O)NC1CCC2=CC(=C(C(=C2C3=CC=C(C(=O)C=C13)OC)OC)OC)OC',
 'CN(CCCl)CCCl.Cl',
 'C1=CC=C(C=C1)C=CC(=O)C2=CC=CC=N2',
 'C1=CN=CC=C1C=CC(=O)O',
 'CN(C)C1=C(C=C(C=C1)C(=O)C2=CC(=C(C=C2)N(C)C)N)N',
 'C1=CC=C(C=C1)C(C2=C(C3=C(C=CC=N3)C=C2)O)NC4=CC=C(C=C4)[N+](=O)[O-]',
 'C1=CC=C(C=C1)C(C2=C(C3=C(C=CC=N3)C=C2)O)NC4=CC=C(C=C4)C(=O)O',
 'C1=CC=C(C=C1)C(C2

In [35]:
def get_morgan_fingerprint(SMILES):
    # Initialize parser parameters
    params = Chem.SmilesParserParams()
    params.useChirality = True  # Preserve stereochemistry information
    params.removeHs = False  # Keep hydrogen atoms
    mfp = []

    for smi in SMILES:
        mol = None
        # Sanitization attempt strategies
        sanitize_attempts = [
            {"sanitize": True},  # First try with standard sanitization
            {
                "sanitize": False,
                "partial_sanitize": True,
            },  # Fallback: partial sanitization
        ]

        for attempt in sanitize_attempts:
            try:
                # Update parameters for this attempt
                current_params = Chem.SmilesParserParams()
                current_params.sanitize = attempt["sanitize"]
                current_params.useChirality = params.useChirality
                current_params.removeHs = params.removeHs

                # Molecule object creation
                mol = Chem.MolFromSmiles(smi, params=current_params)

                if mol and "partial_sanitize" in attempt:
                    # Perform partial sanitization (skip property validation)
                    Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES)

                if mol:  # Successfully processed molecule
                    # Generate Morgan fingerprint
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
                    mfp.append(np.array(fp))
                    break  # Exit attempt loop on success

            except Exception as e:
                if attempt == sanitize_attempts[-1]:  # Final attempt failed
                    print(f"Processing failed: {smi}")
                    print(f"Error details: {str(e)}")
                continue  # Try next attempt

        if not mol:  # All attempts failed
            print(f"Complete processing failure: {smi}")
            mfp.append(np.zeros(2048))  # Insert zero-vector placeholder

    return np.array(mfp)

In [36]:
mfp = get_morgan_fingerprint(SMILES)
mfp = pd.DataFrame(mfp, index=drug_response.index)

[15:22:12] Explicit valence for atom # 4 Sn, 5, is greater than permitted


In [37]:
drug_sim = normalize_similarity_matrix(mfp)
drug_sim

,1,17,89,185,295,353,384,596,629,721,...,829498,830173,830201,830202,830368,830912,831147,831270,831271,831453
1,1.000000,0.846911,0.866893,0.834946,0.874899,0.767472,0.747733,0.874899,0.902982,0.826979,...,0.767472,0.638082,0.700556,0.661452,0.684892,0.700556,0.815043,0.696637,0.704477,0.638082
17,0.846911,1.000000,0.834946,0.779339,0.858894,0.767472,0.771426,0.819019,0.854898,0.811068,...,0.728042,0.614779,0.684892,0.645864,0.661452,0.661452,0.767472,0.673163,0.657552,0.607027
89,0.866893,0.834946,1.000000,0.799155,0.902982,0.779339,0.759571,0.830961,0.866893,0.822998,...,0.747733,0.673163,0.720180,0.665354,0.688805,0.673163,0.819019,0.700556,0.669258,0.641972
185,0.834946,0.779339,0.799155,1.000000,0.807095,0.692720,0.688805,0.783298,0.842921,0.759571,...,0.684892,0.603154,0.657552,0.626422,0.626422,0.641972,0.787259,0.653654,0.653654,0.595413
295,0.874899,0.858894,0.902982,0.807095,1.000000,0.771426,0.759571,0.838933,0.882913,0.838933,...,0.755623,0.657552,0.735913,0.696637,0.696637,0.688805,0.803124,0.708400,0.684892,0.657552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,0.700556,0.661452,0.673163,0.641972,0.688805,0.622539,0.603154,0.657552,0.692720,0.634193,...,0.638082,0.541437,0.572235,0.541437,0.564524,1.000000,0.653654,0.552972,0.568379,0.526083
831147,0.815043,0.767472,0.819019,0.787259,0.803124,0.696637,0.692720,0.763521,0.815043,0.747733,...,0.688805,0.630307,0.684892,0.630307,0.630307,0.653654,1.000000,0.688805,0.649758,0.622539
831270,0.696637,0.673163,0.700556,0.653654,0.708400,0.641972,0.661452,0.653654,0.696637,0.645864,...,0.579953,0.568379,0.599282,0.545280,0.545280,0.552972,0.688805,1.000000,0.572235,0.552972
831271,0.704477,0.657552,0.669258,0.653654,0.684892,0.610902,0.583815,0.677071,0.720180,0.669258,...,0.595413,0.537596,0.599282,0.552972,0.576093,0.568379,0.649758,0.572235,1.000000,0.560672


In [38]:
os.makedirs("../nci_data/drug_sim", exist_ok=True)
chunks = np.array_split(drug_sim, 25)

for i, chunk in tqdm(enumerate(chunks)):
    chunk.to_parquet(
        f"../nci_data/drug_sim/drug_sim_part_{i}.parquet", compression="gzip"
    )

/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
25it [01:10,  2.80s/it]


In [39]:
# # How to read
# def natural_sort_key(s):
#     return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', s)]

# file_paths = glob.glob("../nci_data/drug_sim/drug_sim_part_*.parquet")
# sorted_file_paths = sorted(file_paths, key=natural_sort_key)

# drug_sim_reconstructed = pd.concat([pd.read_parquet(file) for file in tqdm(sorted_file_paths)])

In [40]:
print("Min:", np.min(drug_sim.values))
print("Max:", np.max(drug_sim.values))
print("Mean:", np.mean(drug_sim.values))
print("Median:", np.median(drug_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.701278030905905
Median: 0.708399735004468


# Unified Graph

In [41]:
A_cg

,A2M,AAK1,ABCA1,ABCB1,ABCC1,ABCC5,ABL1,ABL2,ABRACL,ACACB,...,ZNF207,ZNF22,ZNF580,ZNF593,ZNF706,ZNHIT1,ZNHIT3,ZNRD1,ZWINT,ZYX
MCF7,0.000000,0.000000,0.000000,0.00000,0.000000,0.125364,0.000000,0.000000,0.000000,0.521956,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.381257,0.000000,0.000000,0.074835,0.000000
MDA_MB_231,0.000000,0.074879,0.000000,0.00000,0.000000,0.025165,0.000000,0.207760,0.325266,0.000000,...,0.000000,0.427873,0.000000,0.000000,0.660697,0.000000,0.259824,0.121694,0.157818,0.000000
HS578T,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.217541
BT_549,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.982954,0.942017,0.000000,0.000000,...,0.000000,0.000000,0.545743,0.433397,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
T47D,0.000000,0.000000,0.000000,0.00000,0.000000,0.499124,0.000000,0.000000,0.000000,0.000000,...,0.160511,0.000000,0.000000,0.000000,0.000000,0.000000,0.400578,0.075823,0.157252,0.000000
SF_268,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.042573,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
SF_295,0.000000,1.000000,0.009987,0.00000,0.489147,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.870091,0.000000,0.370356,0.651450,0.002774,1.000000,0.693326,0.000000,0.083077,0.498418
SF_539,0.000000,0.380551,0.000000,0.00000,0.335883,0.000000,1.000000,0.541579,0.000000,0.000000,...,0.012623,0.000000,0.094429,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.277290
SNB_19,0.000000,0.000000,0.139962,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.073267,0.000000,0.000000,0.154560,0.000000,0.000000,0.000000,0.000000
SNB_75,1.000000,0.303592,0.215601,0.00000,0.248309,0.000000,0.601546,0.088448,0.000000,0.000000,...,0.000000,0.000000,0.007073,0.000000,0.000000,0.070692,0.000000,0.000000,0.000000,0.062057


In [42]:
indexes = list(A_dc.index) + list(A_cg.index) + list(A_dg.columns)
n_all = len(indexes)
base = pd.DataFrame(np.zeros([n_all, n_all]), index=indexes, columns=indexes)
base

,1,17,89,185,295,353,384,596,629,721,...,ZNF207,ZNF22,ZNF580,ZNF593,ZNF706,ZNHIT1,ZNHIT3,ZNRD1,ZWINT,ZYX
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
295,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNHIT1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNHIT3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNRD1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZWINT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
base.loc[A_cg.index, A_cg.columns] = A_cg
base.loc[A_cg.columns, A_cg.index] = A_cg.T
base.loc[A_dc.index, A_dc.columns] = A_dc
base.loc[A_dc.columns, A_dc.index] = A_dc.T
base.loc[A_dg.index, A_dg.columns] = A_dg
base.loc[A_dg.columns, A_dg.index] = A_dg.T

In [44]:
np.save(
    "../nci_data/idxs.npy",
    pd.DataFrame([list(range(len(base.index))), base.index]).values,
)

In [45]:
edge_index = np.array(base.values.nonzero())
edge_attr = np.array(base.values[base.values.nonzero()])

In [47]:
os.makedirs("../nci_data/edge_idxs", exist_ok=True)
chunk_size = 3000000

num_chunks_index = math.ceil(edge_index.shape[1] / chunk_size)
for i in range(num_chunks_index):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, edge_index.shape[1])
    chunk = edge_index[:, start:end]
    np.save(f"../nci_data/edge_idxs/edge_idxs_chunk_{i}.npy", chunk)

In [48]:
os.makedirs("../nci_data/edge_attrs", exist_ok=True)
chunk_size = 6000000

num_chunks_attr = math.ceil(edge_attr.shape[0] / chunk_size)
for i in range(num_chunks_attr):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, edge_attr.shape[0])
    chunk = edge_attr[start:end]
    np.save(f"../nci_data/edge_attrs/edge_attr_chunk_{i}.npy", chunk)

In [49]:
# # How to read
# def load_and_combine_chunks(pattern, axis=0):
#     chunk_files = sorted(
#         glob.glob(pattern), key=lambda x: int(x.split("_")[-1].split(".")[0])
#     )

#     chunks = [np.load(f) for f in chunk_files]
#     return np.concatenate(chunks, axis=axis)


# edge_index = load_and_combine_chunks("../nci_data/edge_idxs/*.npy", axis=1)
# edge_attr = load_and_combine_chunks("../nci_data/edge_attrs/*.npy", axis=0)